# Attrition risk model — starter notebook

This notebook builds a baseline attrition risk model from HRIS data. It is intentionally simple — the goal is a working, auditable model you can trust and explain to HR stakeholders, not the most accurate possible model.

**What this notebook does:**
1. Loads and validates employee data
2. Engineers features relevant to attrition
3. Trains a logistic regression model (interpretable by design)
4. Evaluates model fairness across demographic groups
5. Generates a risk-scored employee export

**Before using this in production:**
- Complete the [Risk Assessment Template](../03-governance/risk-assessment-template.md)
- Get sign-off from Legal, Privacy, and HR Leadership
- Establish a monitoring cadence for model drift and fairness
- Never share individual risk scores with managers without HRBP oversight

---

In [ ]:
# Requirements
# pip install pandas scikit-learn matplotlib seaborn imbalanced-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load data

Replace the synthetic data generator below with your HRIS export.

**Required columns:**

| Column | Type | Description |
|---|---|---|
| `employee_id` | string | Unique identifier (anonymized) |
| `tenure_months` | int | Months at company |
| `tenure_in_role_months` | int | Months in current role |
| `performance_rating` | int | Most recent rating (1–4 or your scale) |
| `manager_change_12m` | bool | Had a manager change in last 12 months |
| `internal_move_24m` | bool | Had an internal role change in last 24 months |
| `promotion_12m` | bool | Promoted in last 12 months |
| `active_learning_hours_90d` | float | LMS hours in last 90 days |
| `days_since_last_feedback` | int | Days since last performance feedback |
| `compensation_ratio` | float | Employee comp / midpoint of their band |
| `location_type` | str | 'remote', 'hybrid', or 'onsite' |
| `department` | str | Department name |
| `attrited` | bool | **Target:** left the company (1) or stayed (0) |

**Do NOT include:** name, gender, race, age, disability status, or any protected characteristic.

In [ ]:
# ── REPLACE THIS BLOCK WITH YOUR HRIS DATA LOAD ──────────────────────────────
# Example: df = pd.read_csv('hris_export_anonymized.csv')
# Or:      df = pd.read_parquet('workforce_snapshot.parquet')

def generate_synthetic_data(n=2000):
    """Synthetic data for notebook development only. Replace with real data."""
    np.random.seed(RANDOM_STATE)
    df = pd.DataFrame({
        'employee_id': [f'EMP{i:05d}' for i in range(n)],
        'tenure_months': np.random.gamma(shape=3, scale=20, size=n).astype(int).clip(1, 300),
        'tenure_in_role_months': np.random.gamma(shape=2, scale=12, size=n).astype(int).clip(1, 120),
        'performance_rating': np.random.choice([1, 2, 3, 4], size=n, p=[0.05, 0.20, 0.55, 0.20]),
        'manager_change_12m': np.random.choice([0, 1], size=n, p=[0.75, 0.25]),
        'internal_move_24m': np.random.choice([0, 1], size=n, p=[0.70, 0.30]),
        'promotion_12m': np.random.choice([0, 1], size=n, p=[0.85, 0.15]),
        'active_learning_hours_90d': np.random.exponential(scale=5, size=n).round(1),
        'days_since_last_feedback': np.random.randint(1, 365, size=n),
        'compensation_ratio': np.random.normal(loc=1.0, scale=0.15, size=n).clip(0.7, 1.5).round(3),
        'location_type': np.random.choice(['remote', 'hybrid', 'onsite'], size=n, p=[0.50, 0.35, 0.15]),
        'department': np.random.choice(
            ['Engineering', 'Sales', 'Customer Success', 'Marketing', 'People', 'Finance'],
            size=n, p=[0.35, 0.20, 0.15, 0.10, 0.10, 0.10]
        ),
    })
    # Simulate attrition with some signal
    attrition_prob = (
        0.05
        + 0.15 * (df['performance_rating'] == 1)
        + 0.08 * df['manager_change_12m']
        - 0.06 * df['promotion_12m']
        + 0.04 * (df['days_since_last_feedback'] > 180)
        - 0.04 * (df['compensation_ratio'] > 1.1)
        + 0.03 * (df['tenure_in_role_months'] > 36)
        + np.random.normal(0, 0.03, size=n)
    ).clip(0.01, 0.95)
    df['attrited'] = (np.random.random(n) < attrition_prob).astype(int)
    return df

df = generate_synthetic_data()
# ─────────────────────────────────────────────────────────────────────────────

print(f'Dataset: {len(df):,} employees')
print(f'Attrition rate: {df["attrited"].mean():.1%}')
df.head()

## 2. Data validation

In [ ]:
# Check for missing values and basic distributions
print('Missing values:')
print(df.isnull().sum())
print()
print('Class distribution:')
print(df['attrited'].value_counts(normalize=True).round(3))
print()
print('Numeric summary:')
df.describe().round(2)

## 3. Feature engineering

In [ ]:
df_model = df.copy()

# Derived features
df_model['is_low_performer'] = (df_model['performance_rating'] <= 2).astype(int)
df_model['is_high_performer'] = (df_model['performance_rating'] == 4).astype(int)
df_model['long_time_in_role'] = (df_model['tenure_in_role_months'] > 30).astype(int)
df_model['no_recent_feedback'] = (df_model['days_since_last_feedback'] > 120).astype(int)
df_model['underpaid'] = (df_model['compensation_ratio'] < 0.90).astype(int)
df_model['no_recent_learning'] = (df_model['active_learning_hours_90d'] < 1).astype(int)

# Encode location type
df_model = pd.get_dummies(df_model, columns=['location_type'], drop_first=True)
df_model = pd.get_dummies(df_model, columns=['department'], drop_first=True)

FEATURES = [
    'tenure_months', 'tenure_in_role_months', 'performance_rating',
    'manager_change_12m', 'internal_move_24m', 'promotion_12m',
    'active_learning_hours_90d', 'days_since_last_feedback', 'compensation_ratio',
    'is_low_performer', 'is_high_performer', 'long_time_in_role',
    'no_recent_feedback', 'underpaid', 'no_recent_learning',
] + [c for c in df_model.columns if c.startswith('location_type_') or c.startswith('department_')]

X = df_model[FEATURES]
y = df_model['attrited']

print(f'Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features')

## 4. Train model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Logistic regression — chosen for interpretability, not just performance
# If AUC < 0.65, consider gradient boosting (XGBoost/LightGBM) but document the tradeoff
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight=class_weight_dict,
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

# Cross-validation
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='roc_auc')
print(f'CV AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

pipeline.fit(X_train, y_train)
y_prob = pipeline.predict_proba(X_test)[:, 1]
print(f'Test AUC: {roc_auc_score(y_test, y_prob):.3f}')
print()
print(classification_report(y_test, pipeline.predict(X_test)))

## 5. Fairness audit

⚠️ **This section is not optional.** Run it before sharing any scores.

In [ ]:
# Add risk scores to the test set
df_test = df.iloc[X_test.index].copy()
df_test['risk_score'] = y_prob
df_test['risk_tier'] = pd.cut(
    y_prob,
    bins=[0, 0.33, 0.66, 1.0],
    labels=['Low', 'Medium', 'High']
)

# ── REPLACE WITH YOUR ACTUAL DEMOGRAPHIC VARIABLE ────────────────────────────
# If you have a protected characteristic in your test data, audit here.
# Example: df_test['gender'] = ...
# For now we audit by department as a proxy.
audit_var = 'department'
# ─────────────────────────────────────────────────────────────────────────────

print(f'Mean risk score by {audit_var}:')
print(df_test.groupby(audit_var)['risk_score'].agg(['mean', 'count']).round(3))
print()
print(f'High-risk rate by {audit_var}:')
high_risk = (df_test['risk_tier'] == 'High').astype(int)
print(df_test.groupby(audit_var)[audit_var].count().apply(lambda x: x) )

# Flag if any group has >10% higher mean risk score than average
group_means = df_test.groupby(audit_var)['risk_score'].mean()
overall_mean = df_test['risk_score'].mean()
flagged = group_means[group_means > overall_mean + 0.10]
if len(flagged) > 0:
    print(f'\n⚠️  FAIRNESS FLAG: Groups with risk score >10% above average:')
    print(flagged.round(3))
    print('Review before deploying. Consult Legal and People Analytics leadership.')
else:
    print('\n✅ No groups flagged for elevated risk score disparity (>10% threshold).')

## 6. Feature importance

In [ ]:
coef = pipeline.named_steps['model'].coef_[0]
importance_df = pd.DataFrame({
    'feature': FEATURES,
    'coefficient': coef,
    'abs_coefficient': np.abs(coef)
}).sort_values('abs_coefficient', ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#d85a30' if c > 0 else '#1D9E75' for c in importance_df['coefficient']]
ax.barh(importance_df['feature'], importance_df['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature importance (logistic regression coefficients)\nRed = increases attrition risk, Green = decreases', fontsize=12)
ax.set_xlabel('Coefficient value')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Export risk scores

Scores are exported **without names** for HRBP review. HRBPs are responsible for mapping scores to employees and deciding what action, if any, to take.

In [ ]:
# Score all current employees
df_scored = df.copy()

# Re-engineer features for full dataset
# (In production, this would be a function called from a shared module)
df_full = df.copy()
df_full['is_low_performer'] = (df_full['performance_rating'] <= 2).astype(int)
df_full['is_high_performer'] = (df_full['performance_rating'] == 4).astype(int)
df_full['long_time_in_role'] = (df_full['tenure_in_role_months'] > 30).astype(int)
df_full['no_recent_feedback'] = (df_full['days_since_last_feedback'] > 120).astype(int)
df_full['underpaid'] = (df_full['compensation_ratio'] < 0.90).astype(int)
df_full['no_recent_learning'] = (df_full['active_learning_hours_90d'] < 1).astype(int)
df_full = pd.get_dummies(df_full, columns=['location_type'], drop_first=True)
df_full = pd.get_dummies(df_full, columns=['department'], drop_first=True)

# Align columns with training features
X_full = df_full.reindex(columns=FEATURES, fill_value=0)
df_scored['attrition_risk_score'] = pipeline.predict_proba(X_full)[:, 1].round(3)
df_scored['risk_tier'] = pd.cut(
    df_scored['attrition_risk_score'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['Low', 'Medium', 'High']
)

# Export: employee_id and score only — NO names, demographics, or personal data
export_cols = ['employee_id', 'attrition_risk_score', 'risk_tier']
df_export = df_scored[export_cols].sort_values('attrition_risk_score', ascending=False)
df_export.to_csv('attrition_risk_scores.csv', index=False)

print('Risk score distribution:')
print(df_export['risk_tier'].value_counts())
print()
print('Exported to: attrition_risk_scores.csv')
print('⚠️  Share only with authorized HRBPs. Do not distribute to managers directly.')

## Next steps

Before using this model in production:

1. Replace synthetic data with your actual HRIS export
2. Validate feature availability and data quality
3. Run the fairness audit against your actual demographic data (requires secure data environment)
4. Complete the [Risk Assessment Template](../03-governance/risk-assessment-template.md)
5. Establish a retraining schedule (recommend quarterly at minimum)
6. Define HRBP workflow: what action is taken when an employee is 'High' risk?

**Recommended monitoring metrics:**
- Model AUC on held-out data (track quarterly)
- Predicted vs. actual attrition rate (track monthly)
- High-risk rate by department/demographic group (track quarterly for fairness drift)